# 06 Split-GPU + Graphistry

Run llama.cpp on GPU 0 and Graphistry/RAPIDS graph analytics on GPU 1.

**What you will learn:**
- Use `split_gpu_session` to partition GPUs
- Build knowledge graphs with `GraphistryBuilders`
- Visualize graphs with `GraphistrySession` (optional credentials)

**Requirements:** Kaggle T4 x2. Graphistry credentials (optional) for
visualization.

## 1) Install

In [ ]:
!pip -q install git+https://github.com/llamatelemetry/llamatelemetry.git@v0.1.1

## 2) Split GPU session

The context manager pins the LLM workload to one GPU and reserves the
other for graph/analytics tasks.

In [ ]:
from llamatelemetry.kaggle import split_gpu_session

with split_gpu_session(llm_gpu=0, graph_gpu=1) as ctx:
    print("LLM server kwargs:", ctx["llm_server_kwargs"])
    print("Graph GPU ID:", ctx.get("graph_gpu"))

## 3) Build a knowledge graph

`GraphistryBuilders.knowledge_graph()` creates node/edge DataFrames from
entity and relationship lists.

In [ ]:
from llamatelemetry.graphistry import GraphistryBuilders

entities = [
    {"id": "llamatelemetry", "label": "llamatelemetry SDK"},
    {"id": "llama.cpp", "label": "llama.cpp"},
    {"id": "otel", "label": "OpenTelemetry"},
    {"id": "graphistry", "label": "Graphistry"},
]
relationships = [
    {"source": "llamatelemetry", "target": "llama.cpp", "type": "wraps"},
    {"source": "llamatelemetry", "target": "otel", "type": "exports_to"},
    {"source": "llamatelemetry", "target": "graphistry", "type": "visualizes_with"},
]

nodes_df, edges_df = GraphistryBuilders.knowledge_graph(entities, relationships)
print("Nodes:")
print(nodes_df)
print("\nEdges:")
print(edges_df)

## 4) Visualize with Graphistry (optional)

Requires Graphistry credentials stored in Kaggle Secrets:
- `GRAPHISTRY_USERNAME`
- `GRAPHISTRY_PASSWORD`
- `GRAPHISTRY_SERVER` (optional, defaults to hub.graphistry.com)

In [ ]:
# from llamatelemetry.graphistry import GraphistrySession
#
# session = GraphistrySession.from_kaggle_secrets()
# g = session.connector.create_graph(edges_df, nodes_df=nodes_df)
# g.plot()